# Dijet reco-to-gen closures

This notebook follows `macro/plotMcClosures.C::plotDiJetClosures`: it projects the configured $p_T^{ave}$ interval, rebins eta, unit-normalizes the full $\eta_{CM}^{dijet}$ distributions with ROOT `TH1::Scale`, and constructs unnormalized forward/backward ratios with ROOT `TH1::Divide`. Each lower panel is the corresponding curve divided by nominal Gen.

Add another reco-like or smeared distribution by appending one `DijetClosureCurve` below. Its CM, Forward, and Backward histogram-key templates must share the same eta-cut index.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path('/Users/gnigmat/work/cms/jetAnalysis')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import ROOT
except ModuleNotFoundError:
    for path in (Path('/opt/homebrew/lib/python3.14/site-packages'),
                 Path('/opt/homebrew/Cellar/root/6.40.02_1/lib/root')):
        if path.exists() and str(path) not in sys.path:
            sys.path.insert(0, str(path))
    import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    DIJET_DELTA_PHI_SELECTION_LABEL, DIJET_PTAVE_BINS,
)
from hist_analysis.python.dijet_closures import (
    DijetClosureCurve, build_dijet_gen_comparisons,
)
from hist_analysis.python.histogram_io import (
    resolve_combined_file, resolve_direction_file,
)
from hist_analysis.python.plotting import draw_closure
from hist_analysis.python.root_style import (
    DEFAULT_PLOT_STYLE, draw_text_block, save_canvas, set_1d_style,
    set_legend_style, set_pad_style, style_single_panel_axes,
)

## Configuration

The defaults reproduce the macro's nominal Gen/Reco comparison for 60–90 GeV and select the 1.7 jet-eta cut by its canonical macro index. Set `ETA_CUT_INDICES = range(len(ETA_CUTS))` to process all seven cuts. `NORMALIZATION = 'integral'` reproduces the macro's per-bin unit-area shape; `bin_width` produces a unit-area density. `RATIO_OPTION = 'B'` preserves the macro default. Use `''` for standard independent-error propagation if Forward and Backward are treated as independent weighted samples; binomial errors are not generally justified for an arbitrary ratio of independent histograms.

In [ ]:
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'combined'        # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.5)
ETA_CUT_INDICES = (1, 3, 5, 6,)       # 1.9; use range(len(ETA_CUTS)) for all cuts
REBIN_ETA = 2
NORMALIZATION = 'integral'   # macro default: none, integral, or bin_width
RATIO_OPTION = ''            # macro default; use '' for independent errors
FULL_RATIO_RANGE = (0.75, 1.25)
FB_RATIO_RANGE = (0.75, 1.3)
FB_DOUBLE_RATIO_RANGE = (0.75, 1.25)
SAVE_PNG = False
DRAW_GRID = True
OUTPUT_DIR = PROJECT_ROOT / 'hist_analysis' / 'output' / 'dijet_reco_to_gen_closures'

CURVES = (
    DijetClosureCurve(
        'Reco', 'hRecoDijetPtEtaCM_{eta_cut_index}',
        'hRecoDijetPtEtaForward_{eta_cut_index}',
        'hRecoDijetPtEtaBackward_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'Gen', 'hGenDijetPtEtaCM_{eta_cut_index}',
        'hGenDijetPtEtaForward_{eta_cut_index}',
        'hGenDijetPtEtaBackward_{eta_cut_index}',
    ),
    # Examples supported by plotDiJetClosures:
    # DijetClosureCurve(
    #     'Ref', 'hRefDijetPtEtaCM_{eta_cut_index}',
    #     'hRefDijetPtEtaForward_{eta_cut_index}',
    #     'hRefDijetPtEtaBackward_{eta_cut_index}'),
    # DijetClosureCurve('Reco JER x1.0', 'hRecoDijetPtEtaCMJerDef_{eta_cut_index}',
    #     'hRecoDijetPtEtaForwardJerDef_{eta_cut_index}',
    #     'hRecoDijetPtEtaBackwardJerDef_{eta_cut_index}'),
)

In [ ]:
def mc_file(generator, direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, generator, FILE_STEM)
    return resolve_direction_file(BASE_DIR, generator, direction, FILE_STEM)

INPUT_FILE = mc_file(GENERATOR, DIRECTION)
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Missing configured ROOT file: {INPUT_FILE}')
INPUT_FILE

## Build projections and ROOT ratios

Forward and backward projections are deliberately not normalized before division. With the default `integral` mode, each full eta shape has a unit sum of in-range bin contents, matching the macro. `bin_width` instead makes the bin-width-weighted integral equal to one.

In [ ]:
closure_results = {}

for eta_cut_index in ETA_CUT_INDICES:
    if eta_cut_index < 0 or eta_cut_index >= len(ETA_CUTS):
        raise IndexError(f'Invalid eta-cut index: {eta_cut_index}')
    eta_cut = ETA_CUTS[eta_cut_index]
    eta_x_range = (-eta_cut - 0.1, eta_cut + 0.1)
    fb_x_range = (0.0, eta_cut + 0.1)
    for ptave_range in DIJET_PTAVE_BINS:
        eta_shapes, fb_ratios, keys = build_dijet_gen_comparisons(
            INPUT_FILE, CURVES, eta_cut_index=eta_cut_index,
            ptave_range=ptave_range, nominal='Gen',
            rebin_eta=REBIN_ETA, normalization=NORMALIZATION,
            ratio_option=RATIO_OPTION,
        )
        eta_cut_tag = int(round(10.0 * eta_cut))
        ptave_tag = f'{ptave_range[0]:g}_{ptave_range[1]:g}'.replace('.', 'p')
        selection_tag = f'{GENERATOR}_{DIRECTION}_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
        full_output_tag = (
            f'{GENERATOR}_{DIRECTION}_full_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
        )
        fb_output_tag = (
            f'{GENERATOR}_{DIRECTION}_fb_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
        )
        annotations = (
            GENERATOR.capitalize(),
            'Gen and Reco dijets',
            'CM frame',
            f'{ptave_range[0]:g} < p_{{T}}^{{ave}} < {ptave_range[1]:g} GeV',
            f'|#eta_{{CM}}^{{jet}}| < {eta_cut:g}',
            'p_{T}^{Lead} > 50 GeV',
            'p_{T}^{SubLead} > 40 GeV',
            DIJET_DELTA_PHI_SELECTION_LABEL,
        )

        eta_y_title = {
            'none': 'Dijets / bin',
            'integral': 'Fraction of dijets / bin',
            'bin_width': '1/N dN/d#eta_{CM}^{dijet}',
        }[NORMALIZATION]
        eta_canvas, eta_to_gen = draw_closure(
            eta_shapes, 'Gen', title='',
            x_title='#eta_{CM}^{dijet}', y_title=eta_y_title,
            ratio_range=FULL_RATIO_RANGE, x_range=eta_x_range,
            annotations=annotations, grid=DRAW_GRID, headroom=1.6,
            draw_nominal_ratio=False,
            style_indices={'Reco': 0, 'Gen': 1, 'Ref' : 5},
            output=OUTPUT_DIR / f'{full_output_tag}.pdf', save_png=SAVE_PNG,
            canvas_name=full_output_tag,
        )
        fb_canvas, fb_to_gen = draw_closure(
            fb_ratios, 'Gen', title='',
            x_title='#eta_{CM}^{dijet}', y_title='Forward / Backward',
            ratio_range=FB_DOUBLE_RATIO_RANGE, x_range=fb_x_range,
            y_range=FB_RATIO_RANGE,
            annotations=annotations, grid=DRAW_GRID, headroom=1.6,
            draw_nominal_ratio=False,
            style_indices={'Reco': 0, 'Gen': 1, 'Ref' : 5},
            output=OUTPUT_DIR / f'{fb_output_tag}.pdf',
            save_png=SAVE_PNG, canvas_name=fb_output_tag,
        )
        closure_results[selection_tag] = {
            'eta_shapes': eta_shapes, 'eta_to_gen': eta_to_gen,
            'forward_backward': fb_ratios, 'forward_backward_to_gen': fb_to_gen,
            'eta_canvas': eta_canvas, 'forward_backward_canvas': fb_canvas,
            'keys': keys,
        }
        print(selection_tag, keys)

In [ ]:
# For each configured curve and pTave interval, compare F/B across eta cuts.
forward_backward_eta_cut_overlay_results = {}
if not ETA_CUT_INDICES:
    raise ValueError('ETA_CUT_INDICES must contain at least one eta-cut index')
invalid_eta_cut_indices = [
    index for index in ETA_CUT_INDICES if index < 0 or index >= len(ETA_CUTS)
]
if invalid_eta_cut_indices:
    raise IndexError(f'Invalid eta-cut indices: {invalid_eta_cut_indices}')
overlay_fb_x_range = (
    0.0, max(ETA_CUTS[index] for index in ETA_CUT_INDICES) + 0.1,
)

for curve in CURVES:
    curve_tag = ''.join(character.lower() if character.isalnum() else '_'
                        for character in curve.label).strip('_')
    for ptave_range in DIJET_PTAVE_BINS:
        ptave_tag = f'{ptave_range[0]:g}_{ptave_range[1]:g}'.replace('.', 'p')
        output_tag = (
            f'{GENERATOR}_{DIRECTION}_{curve_tag}_fb_etaCutOverlay_ptave_{ptave_tag}'
        )
        canvas = ROOT.TCanvas(
            f'c_{output_tag}', '',
            DEFAULT_PLOT_STYLE.canvas_width, DEFAULT_PLOT_STYLE.canvas_height,
        )
        set_pad_style(canvas, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
        canvas.SetLeftMargin(DEFAULT_PLOT_STYLE.single_panel_left_margin)
        canvas.SetBottomMargin(DEFAULT_PLOT_STYLE.single_panel_bottom_margin)

        legend_height = 0.045 * len(ETA_CUT_INDICES)
        legend = ROOT.TLegend(0.66, 0.88 - legend_height, 0.88, 0.88)
        set_legend_style(legend)
        overlay_histograms = {}

        for style_index, eta_cut_index in enumerate(ETA_CUT_INDICES):
            eta_cut = ETA_CUTS[eta_cut_index]
            eta_cut_tag = int(round(10.0 * eta_cut))
            selection_tag = (
                f'{GENERATOR}_{DIRECTION}_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
            )
            source_histogram = closure_results[selection_tag]['forward_backward'][curve.label]
            histogram_name = (
                f'h_{output_tag}_etaCM_{eta_cut_tag}'
            )
            histogram = source_histogram.Clone(histogram_name)
            histogram.SetDirectory(0)
            histogram.SetTitle('')
            histogram.GetXaxis().SetTitle('#eta_{CM}^{dijet}')
            histogram.GetYaxis().SetTitle('Forward / Backward')
            histogram.GetXaxis().SetRangeUser(*overlay_fb_x_range)
            histogram.GetYaxis().SetRangeUser(*FB_RATIO_RANGE)
            set_1d_style(histogram, style_index)
            style_single_panel_axes(histogram)
            histogram.Draw('E1' if style_index == 0 else 'E1 SAME')
            legend.AddEntry(histogram, f'|#eta_{{CM}}^{{jet}}| < {eta_cut:g}', 'p')
            overlay_histograms[eta_cut_index] = histogram

        legend.Draw()
        annotations = draw_text_block(canvas, (
            GENERATOR.capitalize(),
            f'{curve.label} dijets',
            'CM frame',
            f'{ptave_range[0]:g} < p_{{T}}^{{ave}} < {ptave_range[1]:g} GeV',
            'p_{T}^{Lead} > 50 GeV',
            'p_{T}^{SubLead} > 40 GeV',
            DIJET_DELTA_PHI_SELECTION_LABEL,
        ))
        canvas.Modified()
        canvas.Update()
        save_canvas(
            canvas, OUTPUT_DIR / f'{output_tag}.pdf', save_png=SAVE_PNG,
        )
        canvas._forward_backward_eta_cut_overlay_objects = [
            legend, *annotations, *overlay_histograms.values(),
        ]
        result_key = (curve.label, ptave_range)
        forward_backward_eta_cut_overlay_results[result_key] = {
            'canvas': canvas, 'histograms': overlay_histograms,
        }
        display(canvas)

## Inspect numerical results

The retained dictionaries contain the ROOT histograms and canvases for interactive inspection. This cell reports the ordinary and bin-width-weighted integrals plus the extrema of each non-gen curve's two ratios to Gen.

In [ ]:
for tag, result in closure_results.items():
    print(f'\n{tag}')
    for label in (curve.label for curve in CURVES if curve.label != 'Gen'):
        eta_ratio = result['eta_to_gen'][label]
        fb_ratio = result['forward_backward_to_gen'][label]
        eta_values = [eta_ratio.GetBinContent(i) for i in range(1, eta_ratio.GetNbinsX() + 1)
                      if eta_ratio.GetBinContent(i) != 0.0]
        fb_values = [fb_ratio.GetBinContent(i) for i in range(1, fb_ratio.GetNbinsX() + 1)
                     if fb_ratio.GetBinContent(i) != 0.0]
        print(label, {
            'bin_sum': result['eta_shapes'][label].Integral(),
            'width_integral': result['eta_shapes'][label].Integral('width'),
            'eta/gen range': (min(eta_values), max(eta_values)) if eta_values else None,
            '(F/B)/(F/B)_gen range': (min(fb_values), max(fb_values)) if fb_values else None,
        })